# Step 11: Price-level aggregation on QQQ

This notebook accompanies Lectures 1–2 of *Systematic Trading Notes* and verifies their claims
against real data rather than the lectures' constructed examples. It covers two separate
operations on an OHLCV bar series: collapsing one bar's open/high/low/close into a single
representative price, and combining several bars' prices into a smoothed estimate of the
underlying, less noisy price level. The first is a compression; the second is a causal filter.
Where a method already has a tested implementation in `trading_models`, that implementation is
used directly (Kaufman's Adaptive Moving Average, the Kalman filter's trend state); where none
exists (a plain SMA, EMA, VWAP, and the Kalman filter's level state, which the package's own
function does not expose), the computation is written locally in this notebook and marked as
such.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from trading_models.data.ohlc import load_ohlc
from trading_models.momentum import kama
from trading_models.plotting import SERIES, apply_theme, save_figure
from trading_models.trend_following.data.schema import InstrumentPanel


## 1. Data

An OHLCV bar is a deliberate compression of a trade-by-trade tape into five numbers per period:
the open, the high, the low, the close, and the total volume traded. `trading_models.data.ohlc`
caches open/high/low/close; it does not fetch volume, since none of the project's existing
strategies need it. Volume is fetched separately here, into its own cache file, rather than
changing that tested loader's behaviour for every other caller.


In [2]:
TICKER = "QQQ"
START = "2015-01-01"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"

ohlc = load_ohlc([TICKER], start=START, cache=RAW / "notebook11_qqq_ohlc.csv")
close, high, low, open_ = (ohlc[f][TICKER] for f in ("close", "high", "low", "open"))


def load_volume(ticker: str, start: str, cache: Path) -> pd.Series:
    if cache.exists():
        return pd.read_csv(cache, index_col=0, parse_dates=True)[ticker]
    import yfinance as yf

    raw = yf.download([ticker], start=start, auto_adjust=True, actions=False, progress=False)
    vol = raw["Volume"][ticker]
    cache.parent.mkdir(parents=True, exist_ok=True)
    vol.to_csv(cache)
    return vol


volume = load_volume(TICKER, START, RAW / "notebook11_qqq_volume.csv")

common = close.dropna().index.intersection(volume.dropna().index)
close, high, low, open_, volume = (s.loc[common] for s in (close, high, low, open_, volume))
print(f"{TICKER}: {len(common)} daily bars, {common[0].date()} to {common[-1].date()}")


QQQ: 2960 daily bars, 2015-01-02 to 2026-10-09


In [3]:
apply_theme()

window = close.loc[close.index >= close.index[-1] - pd.Timedelta(days=730)]
fig = go.Figure(
    go.Candlestick(
        x=window.index,
        open=open_.loc[window.index],
        high=high.loc[window.index],
        low=low.loc[window.index],
        close=window,
        name=TICKER,
    )
)
fig.update_layout(title=f"{TICKER}: last two years", height=450, width=1000, xaxis_rangeslider_visible=False)
save_figure(fig, "11_01_candles", ROOT / "figures" / "11_price_aggregation_methods")
fig


## 2. From a bar to a single price

Seven single-bar constructions, each a deterministic function of that bar alone:

| Name | Formula |
|---|---|
| Close | $C$ |
| Open | $O$ |
| Median Price (HL2) | $(H+L)/2$ |
| Typical Price (HLC3) | $(H+L+C)/3$ |
| Weighted Close (HLCC4) | $(H+L+2C)/4$ |
| OHLC4 | $(O+H+L+C)/4$ |
| Body Midpoint | $(O+C)/2$ |

None is more correct than another; each encodes a different judgement about what a bar is
evidence of (the wicks, the body, or both). The difference between them is a function of a
single bar's own range, so it is largest on the single most volatile bar in the sample, found
below exactly rather than chosen by eye.


In [4]:
prices = pd.DataFrame({
    "close": close,
    "open": open_,
    "median_price": (high + low) / 2,
    "typical_price": (high + low + close) / 3,
    "weighted_close": (high + low + 2 * close) / 4,
    "ohlc4": (open_ + high + low + close) / 4,
    "body_mid": (open_ + close) / 2,
})

bar_range_pct = (high - low) / close
event_date = bar_range_pct.idxmax()
print(f"Widest single-bar range: {event_date.date()}, (H-L)/C = {bar_range_pct.loc[event_date]:.2%}")
print(prices.loc[event_date])


Widest single-bar range: 2015-08-24, (H-L)/C = 18.07%
close             90.767723
open              86.868200
median_price      86.319680
typical_price     87.802361
weighted_close    88.543701
ohlc4             87.568821
body_mid          88.817961
Name: 2015-08-24 00:00:00, dtype: float64


In [5]:
zoom = prices.loc[event_date - pd.Timedelta(days=11): event_date + pd.Timedelta(days=11)]
zoom_idx = zoom.index

fig = go.Figure()
fig.add_candlestick(
    x=zoom_idx, open=open_.loc[zoom_idx], high=high.loc[zoom_idx], low=low.loc[zoom_idx],
    close=close.loc[zoom_idx], name=TICKER, opacity=0.5,
)
for col, colour in zip(["median_price", "typical_price", "body_mid"], SERIES[:3], strict=False):
    fig.add_scatter(x=zoom_idx, y=zoom[col], name=col, mode="lines+markers",
                     line=dict(width=1.6, color=colour), marker=dict(size=4))
fig.update_layout(
    title=f"{TICKER}: three price constructions around the widest-range bar ({event_date.date()})",
    height=450, width=1000, xaxis_rangeslider_visible=False,
)
save_figure(fig, "11_02_price_constructions", ROOT / "figures" / "11_price_aggregation_methods")
fig


## 3. Aggregating across bars

Four named weighting schemes plus a structurally different recursive estimator, following
*Systematic Trading Notes*, Lecture 2:

| Method | Weighted by | Needs | Memory |
|---|---|---|---|
| SMA | Equal weight | Price only | Rolling, fixed $N$ |
| EMA | Recency (exponential) | Price only | Expanding (recursive) |
| VWAP | Trading activity | Price and volume | Rolling, fixed $N$ |
| KAMA | Trend-efficiency (adaptive) | Price only | Hybrid |
| Kalman filter | Model confidence, not age | Price only | Expanding, no literal window |

SMA and EMA are one-line pandas calls; VWAP is a small local function; KAMA is imported directly
from `trading_models.momentum`, reusing its tested implementation rather than rewriting it.


In [6]:
N = 20


def vwap(price: pd.Series, volume: pd.Series, window: int) -> pd.Series:
    num = (price * volume).rolling(window).sum()
    den = volume.rolling(window).sum()
    return num / den


sma = close.rolling(N).mean()
ema = close.ewm(span=N, min_periods=N).mean()
vwap_n = vwap(prices["typical_price"], volume, N)
kama_n = kama(close.to_frame(TICKER), er_window=N, fast=2, slow=30)[TICKER]

panel = InstrumentPanel(close=close.to_frame(TICKER))


`kalman_local_linear_trend` is `trading_models`'s own tested function; it returns the filter's
**trend** (slope) state, used elsewhere to drive a directional signal. It does not expose the
**level** state, which is what belongs on the same price-level chart as SMA/EMA/VWAP/KAMA above.
The level-extracting recursion below is therefore written locally, not imported, and is not part
of the tested package; it follows the exact predict/update steps of the real function, just
reading a different element of the same state vector.


In [7]:
from trading_models.trend_following.signals.baselines.statistical import kalman_local_linear_trend

kalman_trend = kalman_local_linear_trend(panel)[TICKER]


def kalman_level(prices: pd.Series, level_var=1e-4, trend_var=1e-6, obs_var=1e-3) -> pd.Series:
    '''Supplementary to trading_models: the filter's level state, not exported by the package's
    own kalman_local_linear_trend (which reads the trend state for a directional signal).'''
    log_p = np.log(prices.to_numpy())
    n = len(log_p)
    out = np.empty(n)
    level, trend = log_p[0], 0.0
    c00, c01, c10, c11 = 1.0, 0.0, 0.0, 1.0
    for t in range(n):
        level, trend = level + trend, trend
        p00, p01, p10, p11 = c00 + c10 + c01 + c11, c01 + c11, c10 + c11, c11
        c00, c01, c10, c11 = p00 + level_var, p01, p10, p11 + trend_var
        innovation = log_p[t] - level
        innovation_var = c00 + obs_var
        g0, g1 = c00 / innovation_var, c10 / innovation_var
        level, trend = level + g0 * innovation, trend + g1 * innovation
        c00, c01, c10, c11 = (1 - g0) * c00, (1 - g0) * c01, c10 - g1 * c00, c11 - g1 * c01
        out[t] = level
    return pd.Series(np.exp(out), index=prices.index)


kalman_lvl = kalman_level(close)


In [8]:
overlay = pd.DataFrame({
    "SMA": sma, "EMA": ema, "VWAP": vwap_n, "KAMA": kama_n, "Kalman level": kalman_lvl,
})

zoom2 = overlay.loc[event_date - pd.Timedelta(days=45): event_date + pd.Timedelta(days=30)]
zoom2_idx = zoom2.index

fig = go.Figure()
fig.add_candlestick(
    x=zoom2_idx, open=open_.loc[zoom2_idx], high=high.loc[zoom2_idx], low=low.loc[zoom2_idx],
    close=close.loc[zoom2_idx], name=TICKER, opacity=0.35,
)
for col, colour in zip(overlay.columns, SERIES, strict=False):
    fig.add_scatter(x=zoom2_idx, y=zoom2[col], name=col, mode="lines", line=dict(width=1.7, color=colour))
fig.update_layout(
    title=f"{TICKER}: five aggregation methods, N={N}, around {event_date.date()}",
    height=480, width=1050, xaxis_rangeslider_visible=False,
    legend=dict(orientation="h", y=1.14, x=0),
    margin=dict(l=55, r=20, t=100, b=40),
)
save_figure(fig, "11_03_aggregation_comparison", ROOT / "figures" / "11_price_aggregation_methods")
fig


## 4. A real event, read through five lenses

The widest-range bar found in Section 2 is also, in general, a high-volume bar: a genuine
market event rather than a constructed one. The table below reads that event, and the several
bars after it, through all five aggregation methods at once, in the same spirit as *Systematic
Trading Notes* Lecture 2 §7 — but on real prices and real volume, not a hand-built series.


In [9]:
event_loc = close.index.get_loc(event_date)
table_idx = close.index[max(0, event_loc - 2): event_loc + 7]

table = pd.DataFrame({
    "close": close.loc[table_idx].round(2),
    "volume": volume.loc[table_idx].astype(int),
    "SMA": sma.loc[table_idx].round(2),
    "EMA": ema.loc[table_idx].round(2),
    "VWAP": vwap_n.loc[table_idx].round(2),
    "KAMA": kama_n.loc[table_idx].round(2),
    "Kalman level": kalman_lvl.loc[table_idx].round(2),
})
table


,close,volume,SMA,EMA,VWAP,KAMA,Kalman level
Date,,,,,,,
2015-08-20,98.71,57747300,102.17,101.84,101.98,101.35,100.85
2015-08-21,94.40,97763400,101.77,101.14,101.01,100.59,98.59
2015-08-24,90.77,150829900,101.23,100.15,98.42,99.20,95.75
2015-08-25,90.43,75088300,100.63,99.22,97.67,97.66,93.65
2015-08-26,94.98,104103800,100.23,98.82,97.07,97.48,93.70
2015-08-27,97.39,69377700,99.94,98.68,96.88,97.48,94.55
2015-08-28,97.37,49919600,99.65,98.56,96.75,97.47,95.19
2015-08-31,96.16,39549400,99.31,98.33,96.59,97.41,95.28
2015-09-01,93.16,73155500,98.83,97.84,96.26,97.14,94.36


## 5. Summary

- Sections 1–2 are a compression (bars, then one price per bar); Section 3 onward is a filter
  (several bars' prices into one smoothed estimate) — the two stay conceptually and
  computationally separate throughout, as in Lecture 2.
- SMA, EMA and VWAP are one-line or near-one-line computations; KAMA and the Kalman filter's
  trend state are imported directly from `trading_models`, so this notebook tests the lecture's
  claims against the project's own tested code, not a reimplementation of it.
- The Kalman filter's level state has no equivalent in the tested package (which only exposes the
  trend state, for a directional signal) and is computed locally here, clearly marked as
  supplementary rather than part of `trading_models` itself.
- Section 4's table and Section 3's chart give the real-data counterpart to Lecture 2's
  constructed worked example: whether the same rolling-window "cliff" and expanding-window smooth
  decay appear on an actual market event is answered directly from the numbers above, not
  asserted.
